# Pipeline: CSV y colección `especies`

**Responsable:** Juan Arias

**Objetivo:** Limpiar el CSV y cargar la colección `especies` según MODELO.md. Se ejecuta antes del notebook 02.

Modelo de referencia: `../MODELO.md`

In [1]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from pymongo import ASCENDING, MongoClient

sys.path.insert(0, "..")
from src.normalizacion import normalize_species

CSV = Path("../data/PanTEon_Database_metadata_v1.6.1.csv")
FASTA = Path("../data/PanTEon_Database_v1.6.1.fasta")
RESULTADOS = Path("../resultados")
RESULTADOS.mkdir(exist_ok=True)

client = MongoClient("mongodb://localhost:27017")
db = client["panteon"]

## 1. Lectura y limpieza

Decisiones (ver `resultados/reporte_calidad_csv.md`): `strip` en todo el texto y vacíos → `null`; los rellenos `No …`/`Not …` de `Order` → `null`; las 44 no-especies marcadas en `Notes` más `Muroidea` llevan `is_species: false`; los 4 comentarios de `Source 2` → `null`; nombres no binomiales y errores taxonómicos se cargan tal cual.

In [2]:
CONSERVAR = ["Kingdom", "Phylum", "Class", "Order", "Family", "Species", "Source", "Source 2", "Notes"]
REINOS = {"Animalia", "Plantae", "Fungi"}
FUENTES = {"Repbase", "Dfam", "RM2+MCH"}
NO_ESPECIE_SIN_MARCA = {"Muroidea"}

df = pd.read_csv(CSV, encoding="utf-8-sig", dtype=str)[CONSERVAR]

# strip en todas las columnas de texto; vacíos -> null
df = df.apply(lambda s: s.str.strip()).replace("", None)
df = df.astype(object).where(df.notna(), None)

# Rellenos de Order -> null
relleno = df["Order"].str.match(r"(?i)^(no|not)\s", na=False)
print("Order con relleno -> null:", relleno.sum())
df.loc[relleno, "Order"] = None

# Comentarios en Source 2 -> null
comentarios = df[df["Source 2"].notna() & ~df["Source 2"].isin(FUENTES)][["Species", "Source 2"]]
print("Source 2 que no son fuentes -> null:", len(comentarios))
df.loc[comentarios.index, "Source 2"] = None

df["is_species"] = ~(
    df["Notes"].str.contains("(?i)no especie", na=False) | df["Species"].isin(NO_ESPECIE_SIN_MARCA)
)
print("is_species = false:", (~df["is_species"]).sum())

otros_reinos = set(df["Kingdom"]) - REINOS
assert not otros_reinos, f"Kingdom fuera de {REINOS}: {otros_reinos}. Detener y revisar."
print("Kingdom:", df["Kingdom"].value_counts().to_dict())

Order con relleno -> null: 26
Source 2 que no son fuentes -> null: 4
is_species = false: 45
Kingdom: {'Fungi': 1559, 'Animalia': 780, 'Plantae': 451}


In [3]:
documentos = [
    {
        "species": r["Species"],
        "taxonomy": {
            "kingdom": r["Kingdom"],
            "phylum": r["Phylum"],
            "class": r["Class"],
            "order": r["Order"],
            "family": r["Family"],
        },
        "source": r["Source"],
        "source_2": r["Source 2"],
        "notes": r["Notes"],
        "is_species": bool(r["is_species"]),
    }
    for r in df.to_dict("records")
]
print(len(documentos))
documentos[0]

2790


{'species': 'Abortiporus biennis',
 'taxonomy': {'kingdom': 'Fungi',
  'phylum': 'Basidiomycota',
  'class': 'Agaricomycetes',
  'order': 'Polyporales',
  'family': 'Podoscyphaceae'},
 'source': 'RM2+MCH',
 'source_2': None,
 'notes': 'B-GUT',
 'is_species': True}

## 2. Carga en `especies`

In [4]:
ESQUEMA = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["species", "taxonomy", "is_species"],
        "properties": {
            "species": {"bsonType": "string"},
            "taxonomy": {
                "bsonType": "object",
                "required": ["kingdom"],
                "properties": {
                    "kingdom": {"enum": sorted(REINOS)},
                    "phylum": {"bsonType": ["string", "null"]},
                    "class": {"bsonType": ["string", "null"]},
                    "order": {"bsonType": ["string", "null"]},
                    "family": {"bsonType": ["string", "null"]},
                },
            },
            "source": {"bsonType": "string"},
            "source_2": {"bsonType": ["string", "null"]},
            "notes": {"bsonType": ["string", "null"]},
            "is_species": {"bsonType": "bool"},
        },
    }
}

# Borra y recrea: el notebook se puede ejecutar varias veces con el mismo resultado
db.drop_collection("especies")
db.create_collection("especies", validator=ESQUEMA, validationLevel="strict", validationAction="error")
res = db.especies.insert_many(documentos)
print("Insertados:", len(res.inserted_ids))

Insertados: 2790


## 3. Índices

In [5]:
db.especies.create_index([("species", ASCENDING)], unique=True)
db.especies.index_information()

{'_id_': {'v': 2, 'key': [('_id', 1)]},
 'species_1': {'v': 2, 'key': [('species', 1)], 'unique': True}}

## 4. Verificación

In [6]:
from pymongo.errors import WriteError

assert db.especies.count_documents({}) == len(df) == 2790
print("is_species:", {d["_id"]: d["n"] for d in db.especies.aggregate(
    [{"$group": {"_id": "$is_species", "n": {"$sum": 1}}}])})
print("kingdom:", {d["_id"]: d["n"] for d in db.especies.aggregate(
    [{"$group": {"_id": "$taxonomy.kingdom", "n": {"$sum": 1}}}])})
print("order null:", db.especies.count_documents({"taxonomy.order": None}))
print("order 'No …' restantes:", db.especies.count_documents({"taxonomy.order": {"$regex": "^(No|Not) "}}))

# El validador rechaza un reino inválido
try:
    db.especies.insert_one({"species": "Prueba", "taxonomy": {"kingdom": "Protista"}, "is_species": True})
    print("ERROR: el validador no rechazó el documento")
except WriteError:
    print("Validador OK: rechaza kingdom = 'Protista'")

is_species: {True: 2745, False: 45}
kingdom: {'Animalia': 780, 'Plantae': 451, 'Fungi': 1559}
order null: 55
order 'No …' restantes: 0
Validador OK: rechaza kingdom = 'Protista'


## 5. Cruce con el FASTA

Solo se leen los encabezados (`>PDB…#CLASE/ORDEN/SUPERFAMILIA @Especie`), línea por línea, sin cargar secuencias.

In [7]:
tes_por_especie = Counter()
with FASTA.open() as f:
    for linea in f:
        if linea.startswith(">"):
            tes_por_especie[linea.split("@", 1)[1].strip()] += 1

fasta = pd.DataFrame(tes_por_especie.items(), columns=["species_fasta", "n_tes"])
fasta["llave"] = fasta["species_fasta"].map(normalize_species)

csv = pd.DataFrame(db.especies.find({}, {"_id": 0, "species": 1, "is_species": 1}))
csv["llave"] = csv["species"].map(normalize_species)

print("TEs en el FASTA:", fasta["n_tes"].sum(), "| especies en el FASTA:", len(fasta))
print("Llaves repetidas en el FASTA tras normalizar:", fasta["llave"].duplicated().sum())
duplicados_csv = int(csv["llave"].duplicated().sum())
assert duplicados_csv == 0, "La llave de cruce fusiona especies del CSV"

TEs en el FASTA: 239946 | especies en el FASTA: 2790
Llaves repetidas en el FASTA tras normalizar: 0


In [8]:
fasta_sin_csv = (fasta[~fasta["llave"].isin(csv["llave"])]
                 .sort_values("n_tes", ascending=False)[["species_fasta", "n_tes"]])
csv_sin_fasta = csv[~csv["llave"].isin(fasta["llave"])][["species", "is_species"]].sort_values("species")
tes_sin_id = int(fasta_sin_csv["n_tes"].sum())

print("Especies del FASTA que no están en el CSV:", len(fasta_sin_csv))
print("Especies del CSV sin TEs en el FASTA:", len(csv_sin_fasta))
print("TEs que quedarían sin species_id:", tes_sin_id)

fasta_sin_csv.to_csv(RESULTADOS / "cruce_fasta_sin_csv.csv", index=False)
csv_sin_fasta.to_csv(RESULTADOS / "cruce_csv_sin_fasta.csv", index=False)
fasta_sin_csv.head(20)

Especies del FASTA que no están en el CSV: 0
Especies del CSV sin TEs en el FASTA: 0
TEs que quedarían sin species_id: 0


,species_fasta,n_tes


`TE_count` del CSV frente a los TEs del FASTA. La columna solo se usa para validar: no se guarda en `especies`.

In [9]:
te_count = pd.read_csv(CSV, encoding="utf-8-sig", usecols=["Species", "TE_count"])
te_count["llave"] = te_count["Species"].map(normalize_species)
te_count = te_count.merge(fasta[["llave", "n_tes"]], on="llave", how="left")
te_count["n_tes"] = te_count["n_tes"].fillna(0).astype(int)
te_count_distinto = te_count[te_count["TE_count"] != te_count["n_tes"]][["Species", "TE_count", "n_tes"]]
te_count_coinciden = len(te_count) - len(te_count_distinto)
print(f"TE_count coincide con el FASTA: {te_count_coinciden} de {len(te_count)} especies")
te_count_distinto

TE_count coincide con el FASTA: 2790 de 2790 especies


,Species,TE_count,n_tes


In [10]:
def tabla(df_, columnas):
    if df_.empty:
        return "Ninguna."
    filas = ["| " + " | ".join(columnas) + " |", "|" + " --- |" * len(columnas)]
    filas += ["| " + " | ".join(str(v) for v in r) + " |" for r in df_[columnas].itertuples(index=False)]
    return "\n".join(filas)


texto = f"""# Cruce CSV de especies ↔ FASTA

Generado por `notebooks/01_pipeline_csv_especies.ipynb`. Llave de cruce: `src.normalizacion.normalize_species`.

| Medida | Valor |
| --- | --- |
| TEs en el FASTA | {fasta["n_tes"].sum()} |
| Especies en el FASTA | {len(fasta)} |
| Especies en `especies` | {len(csv)} |
| Especies del FASTA que no están en el CSV | {len(fasta_sin_csv)} |
| Especies del CSV sin TEs en el FASTA | {len(csv_sin_fasta)} |
| TEs que quedarían sin `species_id` | {tes_sin_id} |

Listas completas: `cruce_fasta_sin_csv.csv` y `cruce_csv_sin_fasta.csv`.

## `TE_count` del CSV frente al FASTA

La columna `TE_count` del CSV solo se usa para validar; no se guarda en `especies`.
Coinciden exactamente {te_count_coinciden} de {len(te_count)} especies.

{tabla(te_count_distinto, ["Species", "TE_count", "n_tes"]) if len(te_count_distinto) else "No hay especies con conteo distinto."}

## Causa y solución de los desajustes

Una primera versión de la llave (strip, espacios dobles a uno y minúsculas) dejaba 4 especies sin
pareja y 533 TEs sin `species_id`. Eran los mismos 4 taxones escritos de dos formas: el FASTA abrevia
los rangos infraespecíficos con punto y el CSV sin él.

| FASTA | CSV | TEs |
| --- | --- | --- |
| Beta vulgaris subsp. vulgaris | Beta vulgaris subsp vulgaris | 296 |
| Abrus pulchellus subsp. cantoniensis | Abrus pulchellus subsp cantoniensis | 178 |
| Marchantia polymorpha subsp. montivagans | Marchantia polymorpha subsp montivagans | 48 |
| Coccomyxa sp. obi | Coccomyxa sp obi | 11 |

Solución: `normalize_species` también elimina los puntos antes de comparar. Ningún nombre del CSV
tiene punto y la nueva llave no genera duplicados en el CSV ({duplicados_csv} duplicados), así que
el cambio no fusiona especies distintas. En la base se guarda el nombre original del CSV.

## Especies del FASTA que no están en el CSV

{tabla(fasta_sin_csv, ["species_fasta", "n_tes"])}

## Especies del CSV sin TEs en el FASTA

{tabla(csv_sin_fasta, ["species", "is_species"])}

## Comentarios de `Source 2` pasados a null

{tabla(comentarios, ["Species", "Source 2"])}
"""
(RESULTADOS / "reporte_cruce.md").write_text(texto, encoding="utf-8")

1972